### Logg

Jeg startet arbeidet i Jupyter-notatboken, der hovedmålet var å hente, behandle og lagre energidata fra Elhub API-et. Først utviklet jeg en funksjon som hentet timesdata for både produksjon og forbruk. Jeg brukte requests til API-kallene og pandas til å strukturere datasettet.
I den første delen av arbeidet testet jeg å lagre dataene i en Cassandra-tabell, basert på løsningen vi jobbet med i Assignment 2. Her forsøkte jeg også å trekke ut spesifikke kolonner fra Cassandra ved hjelp av Spark, blant annet for å se hvordan datastrukturen kunne forenkles før videre bruk. Dette førte til at jeg genererte flere mellomtabeller og testuttrekk, som jeg lagret for senere bruk. Selv om det var nyttig for forståelsen, førte dette til at jeg fylte opp lagringskvoten i databasen ganske raskt.

Deretter opprettet jeg en MongoDB-database. Jeg valgte å ha to collections: én for produksjonsdata og én for forbruksdata. Innlasting ble gjort med insert_many. i denne fasen gjorde jeg også flere testinnsettinger, blant annet mens jeg prøvde å hente spesifikke kolonner og mellomresultater fra Spark- og Cassandra-arbeidet. Dette, kombinert med tidligere testdata, førte til at jeg plutselig fikk feilmeldingen:
“you are over your space quota, using 519 MB of 512 MB” for å løse dette problemet sjekka brukt jeg denne linja print(mongo_client.list_database_names()) og sletet data som ikke ska brukes. 

Stream-App

1- MongoDB siden    
På denne siden hentet jeg timesdata fra MongoDB og visualiserte:
- Produksjonsfordeling i et valgt prisområde (pie chart)
- Timesverdier for utvalgte grupper per måned (line plot)
- Her brukte jeg Plotly for å få interaktive grafer. Jeg la også inn caching for å unngå tunge gjenhentinger fra databasen

2- Map-siden 
Denne siden var en av de mer tekniske. Jeg brukte Folium og et GeoJSON-lag med Elspot-områdene NO1–NO5. Når brukeren klikket på kartet, lagret jeg koordinatene i session_state. Jeg la også til en choropleth-visning med gjennomsnittsdata for valgt periode, samt en markering av valgt prisområde. I tillegg hentet jeg høydeinformasjon fra Open-Meteo sin elevation-API når en posisjon ble valgt.

3- Snow Drift
Her implementerte jeg Tabler (2003)-modellen og beregnet snødrift per sesong. Jeg hentet timesvis vind- og temperaturdata fra Open-Meteo, beregnet SWE og Qt, og presenterte resultatene som:
Line plot over sesonger
Interaktiv vindrose utviklet med Plotly
Jeg la også inn spinner (venting) inne i for-løkken for å indikere at appen hentet data for hvert år.

4- Sliding Window Correlation
På denne siden implementerte jeg en skyvevinduanalyse hvor brukeren kan velge:
meteorologisk variabel
energidataset (produksjon eller forbruk)
lag, vindusstørrelse og sentrumspunkt
Her brukte jeg rolling().corr() fra pandas og bygde tre Plotly-grafer vertikalt.

5- Til slutt utviklet jeg SARIMAX-modellen med valgfri eksogene variabler. Jeg bygde en bred klokketabell via pivotering, håndterte manglende verdier og presenterte prognosen med konfidensintervall. Jeg la også inn feilhåndtering, caching og flere spinner-elementer på tunge operasjoner.

I de siste to sidene, brukte jeg Ai mer, fordi det vanskeligere enn de andre delene, og i tillegg til fordi jeg har mange andre eksamen og innleveringa. Jeg er fornøyd fordi jeg tror at fikk til å få alle funkasjonene til å funke, men skulle ønske at jeg hadde mer tid til å tenker mer over andre ditaljer i appen (Design, farger, gjør appen mer fleksibel)
Jeg prøvde å sortere navigasjonsmeny, men det gikk ikke, og grunn til det fordi jeg har hele koden på side. Målet mitt vær å prøv å sortere data basert på energi og været.

### linker 
Github:https://github.com/Adham-alsahli/ind320--Adham-Al-Sahli-
Streamlit: 

### AI usage 
Jeg brukte også ChatGPT aktivt i arbeidet mitt, spesielt i utviklingen av Streamlit-appen. En av oppgavene var å bytte ut figurer laget med Matplotlib og Seaborn med interaktive Plotly-grafer, og her ga ChatGPT konkrete og korrekte kodeforslag som gjorde overgangen rask og problemfri. I tillegg ble ChatGPT svært nyttig for feilsøking av selve appen. Siden hele prosjektet mitt ligger i én stor App.py-fil med mange ulike sider, var det til tider utfordrende å finne små skrivefeil og inkonsistenser som gjorde at appen sluttet å fungere. ChatGPT hjalp meg med å identifisere slike feil umiddelbart, for eksempel manglende parenteser, feil stavede variabelnavn og feil bruk av funksjoner. Dette sparte meg for mye tid og bidro til en mer stabil og ryddig kodebase.

In [ ]:
import pandas as pd
import requests 
from datetime import datetime
import calendar  

def fetch_data(Year_start, year_end, data_sett):
    URL = "https://elhub.no/api/v1/measurements"
    entity_type = "price-areas"
    urls = []
    all_data = []

    for year in range(Year_start, year_end + 1):

        monthlig_ranges =[]
        for month in range(1, 13):
            #første dag i måneden
            start_date = datetime(year, month, 1, 0, 0, 0)

            #siste dag i måneden
            last_day = calendar.monthrange(year, month)[1]
            end_date = datetime(year, month, last_day, 23, 59, 59)

            #formater datoer til ISO 8601 
            start_ISO = start_date.strftime("%Y-%m-%dT%H:%M:%S")+"%2B01"
            end_ISO = end_date.strftime("%Y-%m-%dT%H:%M:%S")+"%2B01"
            monthlig_ranges.append((start_ISO, end_ISO))

        for month_start, month_end in monthlig_ranges:
            url = f"https://api.elhub.no/energy-data/v0/{entity_type}?dataset={data_sett}&startDate={month_start}&endDate={month_end}"
            urls.append(url)

    for url in urls:
        try:
            response = requests.get(url)
            data = response.json()
            print(f"Fetched data from {url}")

            for entry in data['data']:
                attrs = entry.get('attributes', {})

                if "productionPerGroupMbaHour" in attrs:
                        months_data = attrs["productionPerGroupMbaHour"]
                elif "consumptionPerGroupMbaHour" in attrs:
                        months_data = attrs["consumptionPerGroupMbaHour"]
                else:
                        months_data = []
                    
                all_data.extend(months_data)
        except Exception as e:
                print(f"Error fetching data from {url}: {e}")
    return all_data

In [ ]:
data_df = pd.DataFrame(fetch_data(2022, 2024, "PRODUCTION_PER_GROUP_MBA_HOUR"))

Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-01-01T00:00:00%2B01&endDate=2022-01-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-02-01T00:00:00%2B01&endDate=2022-02-28T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-03-01T00:00:00%2B01&endDate=2022-03-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-04-01T00:00:00%2B01&endDate=2022-04-30T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-05-01T00:00:00%2B01&endDate=2022-05-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=PRODUCTION_PER_GROUP_MBA_HOUR&startDate=2022-06-01T00:00:00%2B01&e

In [ ]:
consumption_df = pd.DataFrame(fetch_data(2021, 2024, "CONSUMPTION_PER_GROUP_MBA_HOUR"))
consumption_df.head()

Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-01-01T00:00:00%2B01&endDate=2021-01-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-02-01T00:00:00%2B01&endDate=2021-02-28T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-03-01T00:00:00%2B01&endDate=2021-03-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-04-01T00:00:00%2B01&endDate=2021-04-30T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-05-01T00:00:00%2B01&endDate=2021-05-31T23:59:59%2B01
Fetched data from https://api.elhub.no/energy-data/v0/price-areas?dataset=CONSUMPTION_PER_GROUP_MBA_HOUR&startDate=2021-06-01T00:00:00%

,consumptionGroup,endTime,lastUpdatedTime,meteringPointCount,priceArea,quantityKwh,startTime
0,cabin,2021-01-01T01:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,177071.56,2021-01-01T00:00:00+01:00
1,cabin,2021-01-01T02:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,171335.12,2021-01-01T01:00:00+01:00
2,cabin,2021-01-01T03:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,164912.02,2021-01-01T02:00:00+01:00
3,cabin,2021-01-01T04:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,160265.77,2021-01-01T03:00:00+01:00
4,cabin,2021-01-01T05:00:00+01:00,2024-12-20T10:35:40+01:00,100607,NO1,159828.69,2021-01-01T04:00:00+01:00


In [ ]:
#drop kolonne "meteringPointCount"
consumption_df = consumption_df.drop(columns=["meteringPointCount"])

In [ ]:

print("Produksjonsdata - kolonner:", data_df.columns.tolist())
print("Konsumdata - kolonner:", consumption_df.columns.tolist())

Produksjonsdata - kolonner: ['endTime', 'lastUpdatedTime', 'priceArea', 'productionGroup', 'quantityKwh', 'startTime']
Konsumdata - kolonner: ['consumptionGroup', 'endTime', 'lastUpdatedTime', 'priceArea', 'quantityKwh', 'startTime']


In [ ]:
from cassandra.cluster import Cluster
from pyspark.sql import SparkSession
import os
import sys

# Hadoop / PATH – samme stil som tidligere
os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.1.1"
os.environ["PYSPARK_HADOOP_VERSION"] = "without"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [ ]:
from cassandra.cluster import Cluster
# Koble til Cassandra-klyngen
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

In [ ]:
from pyspark.sql import SparkSession
#settign er for Spark
session.set_keyspace("assignment_2_keyspace")

Spark = SparkSession.builder.appName("CassandraSparkIntegration"). \
    config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.3.0"). \
    config("spark.cassandra.connection.host", "localhost"). \
    config("spark.cassandra.connection.port", "9042"). \
    config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions"). \
    config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog"). \
    getOrCreate()

Spark.range(1).show()


+---+
| id|
+---+
|  0|
+---+



In [ ]:
#konverter pandas dataframe til spark dataframe
spark_data_df = Spark.createDataFrame(data_df)
spark_data_df.show(5)



+--------------------+--------------------+---------+---------------+-----------+--------------------+
|             endTime|     lastUpdatedTime|priceArea|productionGroup|quantityKwh|           startTime|
+--------------------+--------------------+---------+---------------+-----------+--------------------+
|2022-01-01T01:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1291422.4|2022-01-01T00:00:...|
|2022-01-01T02:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1246209.4|2022-01-01T01:00:...|
|2022-01-01T03:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1271757.0|2022-01-01T02:00:...|
|2022-01-01T04:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1204251.8|2022-01-01T03:00:...|
|2022-01-01T05:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1202086.9|2022-01-01T04:00:...|
+--------------------+--------------------+---------+---------------+-----------+--------------------+
only showing top 5 rows



In [ ]:
#vise skjema og kolonner i spark dataframe
spark_data_df.printSchema()
print("Kolonner i Spark DataFrame for produksjonsdata:", spark_data_df.columns)


root
 |-- endTime: string (nullable = true)
 |-- lastUpdatedTime: string (nullable = true)
 |-- priceArea: string (nullable = true)
 |-- productionGroup: string (nullable = true)
 |-- quantityKwh: double (nullable = true)
 |-- startTime: string (nullable = true)

Kolonner i Spark DataFrame for produksjonsdata: ['endTime', 'lastUpdatedTime', 'priceArea', 'productionGroup', 'quantityKwh', 'startTime']


In [ ]:
from pyspark.sql.functions import to_timestamp

#konverterer startTime, endtime og lastupdatedtime til timestamp type

Spark_data_df = (spark_data_df 
                 .withColumn("starttime", to_timestamp("starttime"))
                 .withColumn("endtime", to_timestamp("endtime"))
                 .withColumn("lastupdatedtime", to_timestamp("lastupdatedtime"))
  
                                                                 )



#konverter alle kolonner til lowecase
spark_data_df = spark_data_df.toDF(*[c.lower() for c in spark_data_df.columns])
spark_data_df.show(7)


+--------------------+--------------------+---------+---------------+-----------+--------------------+
|             endtime|     lastupdatedtime|pricearea|productiongroup|quantitykwh|           starttime|
+--------------------+--------------------+---------+---------------+-----------+--------------------+
|2022-01-01T01:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1291422.4|2022-01-01T00:00:...|
|2022-01-01T02:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1246209.4|2022-01-01T01:00:...|
|2022-01-01T03:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1271757.0|2022-01-01T02:00:...|
|2022-01-01T04:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1204251.8|2022-01-01T03:00:...|
|2022-01-01T05:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1202086.9|2022-01-01T04:00:...|
|2022-01-01T06:00:...|2025-02-01T18:02:...|      NO1|          hydro|  1235809.9|2022-01-01T05:00:...|
|2022-01-01T07:00:...|2025-02-01T18:02:...|      NO1|          hydro|  12

In [ ]:
#printe antall rader
print("Antall rader i Spark DataFrame for produksjonsdata:", spark_data_df.count())

Antall rader i Spark DataFrame for produksjonsdata: 657600


In [ ]:
#skriv data til cassandra tabell
spark_data_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode('append') \
    .options(table="production_per_group_hour", keyspace="assignment_2_keyspace") \
    .save()

In [ ]:
#trekker ut spesifikke kolonner fra spark dataframe
selected_data_df = Spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="production_per_group_hour", keyspace="assignment_2_keyspace") \
    .load() \
    .select("pricearea","productiongroup","starttime","quantitykwh")




In [ ]:
from pymongo import MongoClient 
from pymongo.server_api import ServerApi 
import tomllib
with open("C:/Users/adham/Documents/Nmbu/5/ind320/IND320-main/Ind320/.streamlit/secrets.toml", "rb") as f: 
    mongo_cfg = tomllib.load(f) 

mongo_user = mongo_cfg["MongoDB"]["username"] 
mongo_pwd = mongo_cfg["MongoDB"]["pwd"] 

mongo_uri = ( f"mongodb+srv://{mongo_user}:{mongo_pwd}" "@adham.j1syfjw.mongodb.net/?retryWrites=true&w=majority&appName=IND320" ) 

mongo_client = MongoClient(mongo_uri, server_api=ServerApi('1')) 
mongo_client.admin.command("ping") 
print("MongoDB tilkobling OK.") 

mongo_db = mongo_client["IND320_assignment_4"] 
collection = mongo_db["production_data"]

 #convert spark dataframe til pandas dataframe, også records for innsetting i MongoDB
records = selected_data_df.toPandas().to_dict("records") 
resultat = collection.insert_many(records) 

#sjekk insertion ved å sjekk antall dokumenter i collection 
doc_count = collection.count_documents({}) 
print(f"Total documents in collection: {doc_count}") 

#vis noen dokumenter 
print("\nSample document:") 
print(collection.find_one())

MongoDB tilkobling OK.
Total documents in collection: 872953

Sample document:
{'_id': ObjectId('691fa088d9b96762c3a57667'), 'pricearea': 'NO2', 'productiongroup': 'wind', 'starttime': datetime.datetime(2021, 1, 1, 0, 0), 'quantitykwh': 706.206}


In [ ]:
S_consumption_df = Spark.createDataFrame(consumption_df)
S_consumption_df.show(5)

+----------------+--------------------+--------------------+---------+-----------+--------------------+
|consumptionGroup|             endTime|     lastUpdatedTime|priceArea|quantityKwh|           startTime|
+----------------+--------------------+--------------------+---------+-----------+--------------------+
|           cabin|2021-01-01T01:00:...|2024-12-20T10:35:...|      NO1|  177071.56|2021-01-01T00:00:...|
|           cabin|2021-01-01T02:00:...|2024-12-20T10:35:...|      NO1|  171335.12|2021-01-01T01:00:...|
|           cabin|2021-01-01T03:00:...|2024-12-20T10:35:...|      NO1|  164912.02|2021-01-01T02:00:...|
|           cabin|2021-01-01T04:00:...|2024-12-20T10:35:...|      NO1|  160265.77|2021-01-01T03:00:...|
|           cabin|2021-01-01T05:00:...|2024-12-20T10:35:...|      NO1|  159828.69|2021-01-01T04:00:...|
+----------------+--------------------+--------------------+---------+-----------+--------------------+
only showing top 5 rows



In [ ]:
S_consumption_df = (
    S_consumption_df 
    .withColumn("starttime", to_timestamp("starttime"))
    .withColumn("endtime", to_timestamp("endtime"))
    .withColumn("lastupdatedtime", to_timestamp("lastupdatedtime"))
)

#konverter alle kolonner til lowercase
S_consumption_df = S_consumption_df.toDF(*[c.lower() for c in S_consumption_df.columns])
S_consumption_df.show(7)

+----------------+-------------------+-------------------+---------+-----------+-------------------+
|consumptiongroup|            endtime|    lastupdatedtime|pricearea|quantitykwh|          starttime|
+----------------+-------------------+-------------------+---------+-----------+-------------------+
|           cabin|2021-01-01 01:00:00|2024-12-20 10:35:40|      NO1|  177071.56|2021-01-01 00:00:00|
|           cabin|2021-01-01 02:00:00|2024-12-20 10:35:40|      NO1|  171335.12|2021-01-01 01:00:00|
|           cabin|2021-01-01 03:00:00|2024-12-20 10:35:40|      NO1|  164912.02|2021-01-01 02:00:00|
|           cabin|2021-01-01 04:00:00|2024-12-20 10:35:40|      NO1|  160265.77|2021-01-01 03:00:00|
|           cabin|2021-01-01 05:00:00|2024-12-20 10:35:40|      NO1|  159828.69|2021-01-01 04:00:00|
|           cabin|2021-01-01 06:00:00|2024-12-20 10:35:40|      NO1|  160388.17|2021-01-01 05:00:00|
|           cabin|2021-01-01 07:00:00|2024-12-20 10:35:40|      NO1|   162326.8|2021-01-01 

In [ ]:
session.execute("""
    CREATE TABLE IF NOT EXISTS consumption_per_group_hour (
        pricearea TEXT,
        consumptiongroup TEXT,
        starttime TIMESTAMP,
        endtime TIMESTAMP,
        quantitykwh DOUBLE,
        lastupdatedtime TIMESTAMP,
        PRIMARY KEY ((pricearea, consumptiongroup), starttime)
    ) 
""")

In [ ]:
S_consumption_df.write \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_per_group_hour", keyspace="assignment_2_keyspace") \
    .mode('append') \
    .save()

In [ ]:
#lese data fra cassandra tabell for å sjekke om det ble skrevet riktig
consumption_check_df = Spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_per_group_hour", keyspace="assignment_2_keyspace") \
    .load()

consumption_check_df.show(5)
print("Antall rader i Spark DataFrame for konsumdata:", consumption_check_df.count())

+---------+----------------+-------------------+-------------------+-------------------+-----------+
|pricearea|consumptiongroup|          starttime|            endtime|    lastupdatedtime|quantitykwh|
+---------+----------------+-------------------+-------------------+-------------------+-----------+
|      NO1|         primary|2021-01-01 00:00:00|2021-01-01 01:00:00|2024-12-20 10:35:40|   70166.62|
|      NO1|         primary|2021-01-01 01:00:00|2021-01-01 02:00:00|2024-12-20 10:35:40|   69859.11|
|      NO1|         primary|2021-01-01 02:00:00|2021-01-01 03:00:00|2024-12-20 10:35:40|   71648.79|
|      NO1|         primary|2021-01-01 03:00:00|2021-01-01 04:00:00|2024-12-20 10:35:40|   72086.03|
|      NO1|         primary|2021-01-01 04:00:00|2021-01-01 05:00:00|2024-12-20 10:35:40|  74199.914|
+---------+----------------+-------------------+-------------------+-------------------+-----------+
only showing top 5 rows

Antall rader i Spark DataFrame for konsumdata: 876600


In [ ]:
#sende konsumdata til MongoDB
selected_data_df = consumption_check_df.select("pricearea","consumptiongroup","starttime","quantitykwh")

#konverter spark dataframe til pandas dataframe for innsetting i MongoDB
records = selected_data_df.toPandas().to_dict("records")
with open("C:/Users/adham/Documents/Nmbu/5/ind320/IND320-main/Ind320/.streamlit/secrets.toml", "rb") as f: 
    mongo_cfg = tomllib.load(f) 

mongo_user = mongo_cfg["MongoDB"]["username"] 
mongo_pwd = mongo_cfg["MongoDB"]["pwd"] 

mongo_uri = ( f"mongodb+srv://{mongo_user}:{mongo_pwd}" "@adham.j1syfjw.mongodb.net/?retryWrites=true&w=majority&appName=IND320" ) 

mongo_client = MongoClient(mongo_uri, server_api=ServerApi('1')) 
mongo_client.admin.command("ping") 
print("MongoDB tilkobling OK.")
mongo_db = mongo_client["IND320_assignment_4"]
collection = mongo_db["consumption_data"]

#konverter spark dataframe til pandas dataframe, også records for innsetting i MongoDB
records = selected_data_df.toPandas().to_dict("records")

#new records insertion
resultat = collection.insert_many(records)
print(f"Inserted {len(resultat.inserted_ids)} documents into MongoDB collection 'consumption_data'.")

#sjekk insertion ved å sjekk antall dokumenter i collection
doc_count = collection.count_documents({})
print(f"Total documents in collection: {doc_count}")

#vis noen dokumenter
print("\nSample document:")
print(collection.find_one())


MongoDB tilkobling OK.
Inserted 876600 documents into MongoDB collection 'consumption_data'.
Total documents in collection: 876600

Sample document:
{'_id': ObjectId('691fa2a2d9b96762c3b2c861'), 'pricearea': 'NO2', 'consumptiongroup': 'cabin', 'starttime': datetime.datetime(2021, 1, 1, 0, 0), 'quantitykwh': 142641.7}


In [ ]:
print(mongo_client.list_database_names())


['IND320_assignment_4', 'admin', 'local']


In [ ]:
#mongo_client.drop_database("IND320_assignment_4")
print("Droppet database 'IND320_assignment_4'.")

Droppet database 'IND320_assignment_4'.
